## 🚑 Patch: Robust target extraction & safer `_prepare`

Applied on 2025-09-18 19:37:57.

Some rows have `target = ["injured_victim"]` (single element), while older code expected `target[1]`.
This patch:
- Adds `get_target_const(target)` that returns the constant robustly whether `target` is a string,
  a single-element list, or a multi-element list (we take the **last** item).
- Re-defines `_prepare(tokenizer, ds)` to use the helper safely.

**Note**: Run this cell before calling `train(...)`.


In [ ]:
from typing import Any

def get_target_const(target: Any) -> str:
    """
    Return the constant name from `target` robustly.
    - If target is a string -> return it.
    - If target is a list/tuple -> return the LAST element (handles [const] or [prop, const]).
    - Otherwise raise a clear error to help debugging.
    """
    if isinstance(target, str):
        return target
    if isinstance(target, (list, tuple)):
        if len(target) == 0:
            raise ValueError("Empty target encountered; expected at least one element")
        return target[-1]
    raise TypeError(f"Unsupported target type: {type(target)}; value={target}")

def _prepare(tokenizer, ds):
    from datasets import Dataset
    import numpy as np

    def convert(batch):
        all_input_ids, all_attention_mask, all_token_type_ids, all_labels = [], [], [], []

        for sent, prefix, target in zip(batch["sentence"], batch["prefix"], batch["target"]):
            # Robustly get the constant string
            target_const = get_target_const(target)

            # Tokenize the prefix tokens as individual words (space-join if needed)
            prefix_text = " ".join(prefix) if isinstance(prefix, list) else str(prefix)
            encoded = tokenizer(
                prefix_text,
                truncation=True,
                max_length=256,
                padding="max_length",
                return_offsets_mapping=False,
                return_tensors=None,
            )

            input_ids = encoded["input_ids"]
            attention_mask = encoded["attention_mask"]
            token_type_ids = encoded.get("token_type_ids", [0] * len(input_ids))

            # Build label vector same length as input_ids (all zeros by default)
            label_ids = [0] * len(input_ids)
            try:
                toks_wp = tokenizer.convert_ids_to_tokens(input_ids)
                const_lc = str(target_const).lower()
                for j, wp in enumerate(toks_wp):
                    if wp in ("[CLS]", "[SEP]", "[PAD]"):
                        continue
                    w = wp.replace("##", "").lower()
                    if w == const_lc:
                        label_ids[j] = 1
            except Exception:
                pass

            all_input_ids.append(input_ids)
            all_attention_mask.append(attention_mask)
            all_token_type_ids.append(token_type_ids)
            all_labels.append(label_ids)

        return {
            "input_ids": all_input_ids,
            "attention_mask": all_attention_mask,
            "token_type_ids": all_token_type_ids,
            "labels": all_labels,
        }

    keep = {"sentence", "prefix", "prefix_target", "target"}
    return ds.map(
        convert,
        batched=True,
        remove_columns=[c for c in ds.column_names if c not in keep]
    )

print("Patched: get_target_const + safer _prepare loaded.")


# 🔧 Prefix Trimming Patch (auto-inserted)

This notebook was augmented on 
2025-09-18 19:33:41
 to trim signature-prefixes in your dataset.

**What it does**
- For each example with fields `prefix` (list of strings) and `prefix_target` (list of ints),
  it finds the **last** token wrapped in carets (e.g., `<const>`, `<predicates>`),
  then **removes that token and everything before it** from `prefix`.
  It removes the same number of leading elements from `prefix_target` to keep alignment.

**Example**
```
prefix: ["<search_and_rescue>", "<type>", "person", "<const>", "safe_civilian", ..., "injured_victim"]
prefix_target: [0, 0, 0, 0, 0, ..., 1]

→ trimmed prefix: ["safe_civilian", ..., "injured_victim"]
→ trimmed prefix_target: [0, ..., 1]
```

**How to use**
1. Run the cell below to define the trimming functions.
2. Apply them to your data structure (list of dicts, pandas DataFrame, or 🤗 Datasets).
3. Proceed with training as usual; downstream code should now see the trimmed prefixes.


In [ ]:
from __future__ import annotations
import ast
import re
from typing import Any, Dict, List

CARET_RE = re.compile(r"^<[^<>]+>$")

def _coerce_list(x):
    """If a field is a string representation of a list, parse it; otherwise return as-is."""
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        # Attempt safe parsing of Python list literal
        if (s.startswith('[') and s.endswith(']')) or (s.startswith('(') and s.endswith(')')):
            try:
                val = ast.literal_eval(s)
                if isinstance(val, (list, tuple)):
                    return list(val)
            except Exception:
                pass
    return x

def trim_prefix_and_targets(example: Dict[str, Any]) -> Dict[str, Any]:
    """
    Given an example dict with 'prefix' (list[str]) and 'prefix_target' (list[int]),
    trim everything up to and including the **last** caret-wrapped token in 'prefix'.
    Keep 'prefix_target' aligned by dropping the same number of leading entries.
    """
    if not isinstance(example, dict):
        return example
    prefix = _coerce_list(example.get('prefix'))
    target = _coerce_list(example.get('prefix_target'))
    if not isinstance(prefix, list) or not isinstance(target, list):
        return example

    # Find last caret-wrapped token index
    last_idx = None
    for i, tok in enumerate(prefix):
        if isinstance(tok, str) and CARET_RE.match(tok):
            last_idx = i
    if last_idx is None:
        return example  # nothing to trim

    k = last_idx + 1  # remove up to and including the last caret-token
    new_prefix = prefix[k:]
    # If target shorter than k (malformed), fall back to no-op on target
    new_target = target[k:] if len(target) >= k else target

    # Only accept if lengths remain consistent (e.g., classification over remaining entries)
    example['prefix'] = new_prefix
    example['prefix_target'] = new_target
    return example

def trim_list_of_dicts(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [trim_prefix_and_targets(r) for r in records]

def trim_dataframe(df):
    import pandas as pd
    if not hasattr(df, 'copy'):
        return df
    df = df.copy()
    if 'prefix' in df.columns and 'prefix_target' in df.columns:
        df['prefix'] = df['prefix'].apply(_coerce_list)
        df['prefix_target'] = df['prefix_target'].apply(_coerce_list)
        def _row_apply(row):
            ex = {'prefix': row['prefix'], 'prefix_target': row['prefix_target']}
            ex = trim_prefix_and_targets(ex)
            row['prefix'] = ex['prefix']
            row['prefix_target'] = ex['prefix_target']
            return row
        df = df.apply(_row_apply, axis=1)
    return df

def trim_hf_dataset(ds):
    """Map over a HuggingFace Dataset or DatasetDict."""
    try:
        from datasets import Dataset, DatasetDict
    except Exception:
        raise RuntimeError("🤗 Datasets not installed. pip install datasets")
    if isinstance(ds, DatasetDict):
        return ds.map(trim_prefix_and_targets)
    elif isinstance(ds, Dataset):
        return ds.map(trim_prefix_and_targets)
    else:
        raise TypeError("Expected a Dataset or DatasetDict")

print("Prefix trimming utilities loaded. Examples:")
print("- records = trim_list_of_dicts(records)")
print("- df = trim_dataframe(df)")
print("- ds = trim_hf_dataset(ds)")


In [ ]:
# Quick self-test using the user's example
example = {
    "id": 14,
    "prop_id": "prop_2",
    "sentence": ["communicate", "with", "injured", "victim"],
    "target": ["injured_victim"],
    "prefix": [
        "<search_and_rescue>", "<type>", "person", "<const>",
        "safe_civilian", "safe_hostile", "safe_person", "safe_rescuer", "safe_victim",
        "unsafe_civilian", "unsafe_person", "unsafe_rescuer", "unsafe_victim",
        "injured_civilian", "injured_hostile", "injured_person", "injured_rescuer", "injured_victim"
    ],
    "prefix_target": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
}
trimmed = trim_prefix_and_targets(dict(example))
print('Original prefix length:', len(example['prefix']))
print('Trimmed  prefix length:', len(trimmed['prefix']))
print('Original prefix_target length:', len(example['prefix_target']))
print('Trimmed  prefix_target length:', len(trimmed['prefix_target']))
print('Trimmed prefix[:5]:', trimmed['prefix'][:5])
assert trimmed['prefix'][0] == 'safe_civilian'
assert len(example['prefix']) - len(trimmed['prefix']) == 4
assert len(example['prefix_target']) - len(trimmed['prefix_target']) == 4
print('✅ Self-test passed')


In [ ]:

from pathlib import Path
import os, json, hashlib
from typing import List, Dict, Any, Tuple

import numpy as np
from datasets import Dataset, concatenate_datasets
from transformers import (
    BertForTokenClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import torch
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Must match your builder script
MAX_PREFIX_SHARD = 20
MODEL_NAME = "bert-base-uncased"
SEED = 42

# Set your paths here (adjust if needed)
data_dir = Path("grounding_data")  # produced by build_grounding_data.py
output_dir = Path("outputs_joint")  # where we save the single model
epochs = 4
batch = 16
lr = 3e-5
eval_steps = 200
early_stopping = 3
early_stopping_threshold = 1e-4

In [ ]:
from pathlib import Path

base_dir = Path("grounding_data")
out_base = Path("grounding_data_combined")

splits = ["train", "test", "total"]
kinds = ["argument", "predicate"]

out_base.mkdir(exist_ok=True)

for split in splits:
    split_out_dir = out_base / split
    split_out_dir.mkdir(exist_ok=True)

    # Collect all domain filenames that appear in either argument/ or predicate/ for this split
    domain_files = set()
    for kind in kinds:
        for p in (base_dir / kind / split).glob("*.jsonl"):
            domain_files.add(p.name)

    # For each domain, concatenate argument + predicate into the combined dir
    for domain_name in domain_files:
        out_path = split_out_dir / domain_name
        with out_path.open("w", encoding="utf-8") as out_f:
            for kind in kinds:
                in_path = base_dir / kind / split / domain_name
                if in_path.exists():
                    with in_path.open("r", encoding="utf-8") as in_f:
                        for line in in_f:
                            out_f.write(line)


In [ ]:
data_dir = Path('/home/will.english/Desktop/Research/ICLR-2026-GinSign/grounding_data_combined')
output_dir = Path('ICML_BERT_Grounder_Joint')
model_name = 'bert-base-uncased'
epochs = 3
batch = 32
lr = 5e-5
eval_steps = 500  
early_stopping = 3
early_stopping_threshold = 1e-6

In [ ]:
import re
from collections import defaultdict
import random

dirs = ["north", "south", "east", "west", "northwest", "northeast", "southwest", "southeast"]
nums = ["1st", "2nd", "3rd", "4th", "5th", "6th", "7th", "8th", "9th", "10th"]
roads = ["street", "avenue"]
all_roads = [dir+"_"+num+"_"+road for dir in dirs for num in nums for road in roads]

# ---------------------------------------------------------------------
# 1.  Signatures
# ---------------------------------------------------------------------
TYPE_CONSTANTS = {
    "<search_and_rescue>": {
        "person": [
            "safe_civilian","safe_hostile","safe_person","safe_rescuer","safe_victim",
            "unsafe_civilian","unsafe_person","unsafe_rescuer","unsafe_victim",
            "injured_civilian","injured_hostile","injured_person","injured_rescuer",
            "injured_victim"
        ],
        "threat": [
            "active_debris","active_fire_source","active_flood","active_gas_leak",
            "active_unstable_beam","debris","fire_source","flood","gas_leak",
            "impending_debris","impending_fire_source","impending_flood",
            "impending_gas_leak","impending_unstable_beam","inactive_debris",
            "inactive_fire_source","inactive_flood","inactive_gas_leak",
            "inactive_unstable_beam","unstable_beam","nearest_fire_source",
            "nearest_flood","nearest_gas_leak","nearest_unstable_beam",
            "probable_debris","probable_fire_source","probable_flood",
            "probable_gas_leak","probable_unstable_beam","nearest_debris"
        ],
    },
    "<traffic_light>": {
        "light": [
            "light_east","light_north","light_south","light_west"
        ],
        "color": ["green","red","yellow"],
        "lane":   # truncated list – add the full list if you need 100 % coverage
            all_roads
        ,
        "traffic_target": [
            "car","collision","cyclist","motorcycle","pedestrian","person",
            "vehicle","jaywalker"
        ],
    },
    "<warehouse>": {
        "item": [
            "aeroplane","apple","backpack","banana","baseball_bat","baseball_glove",
            "bear","bed","bench","bicycle","bird","boat","book","bottle","bowl",
            "broccoli","bus","cake","car","carrot","cat","cell_phone","chair",
            "clock","cow","cup","dining_table","dog","donut","elephant",
            "fire_hydrant","fork","frisbee","giraffe","hair-drier","handbag",
            "horse","hot_dog","keyboard","kite","knife","laptop","microwave",
            "motorbike","mouse","orange","oven","parking_meter","person",
            "pizza","potted_plant","refrigerator","remote","sandwich","scissors",
            "sheep","sink","skateboard","skis","snowboard","sofa","spoon",
            "sports_ball","stop_sign","suitcase","surfboard","teddy-bear",
            "tennis_racket","tie","toaster","toilet","toothbrush","traffic_light",
            "train","truck","tv_monitor","umbrella","vase","wine_glass","zebra"
        ],
        "location": ["loading_dock","shelf"],
        "ego" : []
    },
}


In [ ]:
from datasets import Dataset, concatenate_datasets
from transformers import BertTokenizerFast, DataCollatorForTokenClassification

def _load_split(split_dir):
    # Expects files at fine_prefix_grounding/{train,test}/{domain}.jsonl
    import json, pathlib
    paths = list(pathlib.Path(split_dir).glob("*.jsonl"))
    dss = []
    for p in paths:
        rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]
        dss.append(Dataset.from_list(rows))
    if not dss:
        raise FileNotFoundError(f"No jsonl files found under {split_dir}")
    return concatenate_datasets(dss)

def _prepare(tokenizer: BertTokenizerFast, ds: Dataset):
    def convert(batch):
        all_input_ids, all_attention_mask, all_token_type_ids, all_labels = [], [], [], []

        for sent, prefix, target in zip(batch["sentence"], batch["prefix"], batch["target"]):
            # get_target_const(target) is the arg constant string (e.g., "injured_victim")
            target_const = get_target_const(target)

            # Build word-level labels over the *prefix* tokens
            # Find the prefix index where the constant appears
            try:
                target_idx = prefix.index(target_const)
            except ValueError:
                # If not found, label all zeros (and let training skip it)
                target_idx = None

            lab = [0] * len(prefix)
            if target_idx is not None:
                lab[target_idx] = 1

            # Tokenize with sentence as first sequence, prefix as second
            enc = tokenizer(
                sent,
                prefix,
                is_split_into_words=True,  # both are token lists
                truncation=True,
                # no padding here (use DataCollatorForTokenClassification during training)
                return_tensors=None,
            )

            word_ids = enc.word_ids()            # indices into whichever sequence a token came from
            seq_ids  = enc.sequence_ids()        # 0 for sentence, 1 for prefix, None for specials

            # Build token-level labels aligned to tokenizer output
            labels = []
            for wid, sid in zip(word_ids, seq_ids):
                if sid == 1 and wid is not None:
                    # wid here is the word index *within the prefix list*
                    # Ensure it’s in range (defensive guard to avoid IndexError)
                    if 0 <= wid < len(lab):
                        labels.append(lab[wid])
                    else:
                        labels.append(-100)
                else:
                    labels.append(-100)

            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            # Some tokenizers won’t return token_type_ids for RoBERTa, etc.
            if "token_type_ids" in enc:
                all_token_type_ids.append(enc["token_type_ids"])
            else:
                all_token_type_ids.append([0] * len(enc["input_ids"]))
            all_labels.append(labels)

        out = {
            "input_ids": all_input_ids,
            "attention_mask": all_attention_mask,
            "token_type_ids": all_token_type_ids,
            "labels": all_labels,
        }
        return out

    keep = {"sentence", "prefix", "prefix_target", "target"}  # keep target so we can find the constant
    return ds.map(
        convert,
        batched=True,
        remove_columns=[c for c in ds.column_names if c not in keep]
    )


In [ ]:

def train(data_dir: Path, out_dir: Path, *, model_name="bert-base-uncased", epochs=3, batch=8,
          lr=5e-5, eval_steps=500, patience=3, threshold=1e-6):
    tok = BertTokenizerFast.from_pretrained(model_name)
    tr_ds = _prepare(tok, _load_split(data_dir/"train"))
    te_ds = _prepare(tok, _load_split(data_dir/"test"))

    model = BertForTokenClassification.from_pretrained(model_name, num_labels=2)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=eval_steps,
        save_strategy="steps",
        save_steps=eval_steps,
        logging_strategy="steps",
        logging_steps=max(1, eval_steps//5),
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        report_to=[],
    )

    cb = EarlyStoppingCallback(
        early_stopping_patience=patience,
        early_stopping_threshold=threshold,
    )

    Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=te_ds,
        tokenizer=tok,
        data_collator=DataCollatorForTokenClassification(tok),
        callbacks=[cb],
    ).train()

    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)


In [ ]:

train(data_dir, output_dir, model_name=model_name, epochs=epochs, batch=batch,
        lr=lr, eval_steps=eval_steps, patience=early_stopping, threshold=early_stopping_threshold)



In [ ]:
# %% [code]
import numpy as np
from sklearn.metrics import classification_report
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer

# reload tokenizer + model from your output directory
tok = BertTokenizerFast.from_pretrained(output_dir)
model = BertForTokenClassification.from_pretrained(output_dir)

# prepare test dataset again (ensures tokenization/labels are aligned)
test_ds = _prepare(tok, _load_split(data_dir/"test"))

# define a metric computation function
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1).flatten()
    labels = p.label_ids.flatten()

    # filter out ignored labels (-100)
    mask = labels != -100
    preds = preds[mask]
    labels = labels[mask]

    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    # return only top-level scores to Trainer; full report can be printed separately
    return {
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"],
        "accuracy": report["accuracy"],
    }

# set up Trainer for evaluation
trainer = Trainer(
    model=model,
    tokenizer=tok,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorForTokenClassification(tok),
)

# run evaluation
results = trainer.evaluate()
print("=== Test Results ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


In [ ]:
data_dir = Path('/home/will.english/Desktop/Research/ICLR-2026-GinSign/grounding_data_sr+tl/argument')
output_dir = Path('PostSubmission_ICLR_BERT_Grounder_Joint_sr+tl')
out_path = Path(output_dir) / "per_test_metrics_arg_only.json"

import json
from pathlib import Path
import numpy as np
from sklearn.metrics import classification_report
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    Trainer,
    DataCollatorForTokenClassification,
)

# --- reload tokenizer + model from your output directory
tok = BertTokenizerFast.from_pretrained(output_dir)
model = BertForTokenClassification.from_pretrained(output_dir)
collator = DataCollatorForTokenClassification(tok)

# --- metric function (kept compatible with your original "label 1" focus)
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1).flatten()
    labels = p.label_ids.flatten()

    # filter out ignored labels (-100)
    mask = labels != -100
    preds = preds[mask]
    labels = labels[mask]

    report = classification_report(labels, preds, output_dict=True, zero_division=0)
    return {
        "precision": report.get("1", {}).get("precision", 0.0),
        "recall": report.get("1", {}).get("recall", 0.0),
        "f1": report.get("1", {}).get("f1-score", 0.0),
        "accuracy": report.get("accuracy", 0.0),
        # if you want macro F1 too, uncomment:
        # "macro_f1": report.get("macro avg", {}).get("f1-score", 0.0),
    }

# --- utility: discover test sets inside data_dir/test
from pathlib import Path
import json
from datasets import Dataset

def _load_single_jsonl(path: Path) -> Dataset:
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

# --- utility: discover test sets inside data_dir/test
test_root = Path(data_dir) / "test"
test_sets = []

if not test_root.exists():
    raise FileNotFoundError(f"No such directory: {test_root}")

# 1) subdirectories (e.g., test/<domain>/...) -> use your existing loader
for sub in sorted(p for p in test_root.iterdir() if p.is_dir()):
    ds = _prepare(tok, _load_split(sub))
    test_sets.append((sub.name, ds))

# 2) standalone jsonl files directly under test/ -> load file explicitly
for f in sorted(test_root.glob("*.jsonl")):
    ds = _prepare(tok, _load_single_jsonl(f))
    test_sets.append((f.stem, ds))

if not test_sets:
    raise RuntimeError(f"No test datasets found under {test_root}")

# --- single Trainer; pass eval_dataset per call
trainer = Trainer(
    model=model,
    tokenizer=tok,
    compute_metrics=compute_metrics,
    data_collator=collator,
)

all_results = {}

print("=== Per-Dataset Test Results ===")
for name, ds in test_sets:
    metrics = trainer.evaluate(eval_dataset=ds)
    all_results[name] = metrics

    # pretty print a small summary
    print(f"\n-- {name} --")
    for k, v in metrics.items():
        # metrics typically include eval_loss and the keys from compute_metrics
        try:
            print(f"{k}: {float(v):.4f}")
        except Exception:
            print(f"{k}: {v}")

# --- optionally: save raw metrics to disk

with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nSaved per-dataset metrics to: {out_path}")

# --- (optional) full classification reports per dataset
# Uncomment if you also want detailed per-class reports written to disk.
# for name, ds in test_sets:
#     # get predictions to build full report
#     raw = trainer.predict(ds)
#     preds = np.argmax(raw.predictions, axis=-1).flatten()
#     labels = raw.label_ids.flatten()
#     mask = labels != -100
#     report = classification_report(labels[mask], preds[mask], output_dict=True, zero_division=0)
#     with open(Path(output_dir) / f"classification_report_{name}.json", "w") as f:
#         json.dump(report, f, indent=2)


In [ ]:
import os
import json
import glob
import pandas as pd

# If you're running this from the directory that contains the 9 result folders:
BASE_DIR = "."

# Discover all 9 folders
folders = sorted(
    glob.glob(os.path.join(BASE_DIR, "PostSubmission_ICLR_BERT_Grounder_*"))
)

domains = {
    "search_and_rescue": "sr",
    "traffic_light": "tl",
    "warehouse": "wh",
}

# Helper to safely load a JSON file if it exists
def load_json(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None

rows = []

for folder in folders:
    folder_name = os.path.basename(folder)
    
    # Determine model type from folder name
    if "Args_" in folder_name:
        model_type = "args"
    elif "Preds_" in folder_name:
        model_type = "preds"
    elif "Joint_" in folder_name:
        model_type = "joint"
    else:
        model_type = "unknown"
    
    # Initialize row with identifiers
    row = {
        "run": folder_name,
        "model_type": model_type,
    }
    
    # Initialize all metric columns to NaN
    for d_short in domains.values():
        for role in ["pred", "arg"]:
            row[f"{d_short}_{role}_acc"] = float("nan")
            row[f"{d_short}_{role}_f1"] = float("nan")
    
    # Load the appropriate metrics JSONs
    if model_type in ["args", "preds"]:
        metrics_path = os.path.join(folder, "per_test_metrics.json")
        metrics = load_json(metrics_path)
        if metrics is None:
            print(f"Warning: missing {metrics_path}")
            rows.append(row)
            continue
        
        for dom_full, d_short in domains.items():
            if dom_full not in metrics:
                continue
            acc = metrics[dom_full].get("eval_accuracy", float("nan"))
            f1  = metrics[dom_full].get("eval_f1", float("nan"))
            
            if model_type == "args":
                row[f"{d_short}_arg_acc"] = acc
                row[f"{d_short}_arg_f1"]  = f1
            elif model_type == "preds":
                row[f"{d_short}_pred_acc"] = acc
                row[f"{d_short}_pred_f1"]  = f1
    
    elif model_type == "joint":
        pred_metrics_path = os.path.join(folder, "per_test_metrics_pred_only.json")
        arg_metrics_path  = os.path.join(folder, "per_test_metrics_arg_only.json")
        
        pred_metrics = load_json(pred_metrics_path) or {}
        arg_metrics  = load_json(arg_metrics_path) or {}
        
        # Fill predicate-only metrics
        for dom_full, d_short in domains.items():
            if dom_full in pred_metrics:
                acc = pred_metrics[dom_full].get("eval_accuracy", float("nan"))
                f1  = pred_metrics[dom_full].get("eval_f1", float("nan"))
                row[f"{d_short}_pred_acc"] = acc
                row[f"{d_short}_pred_f1"]  = f1
            
            if dom_full in arg_metrics:
                acc = arg_metrics[dom_full].get("eval_accuracy", float("nan"))
                f1  = arg_metrics[dom_full].get("eval_f1", float("nan"))
                row[f"{d_short}_arg_acc"] = acc
                row[f"{d_short}_arg_f1"]  = f1
    
    rows.append(row)

# Build DataFrame
df = pd.DataFrame(rows)

# Order columns: id columns first, then per-domain metrics
metric_cols = []
for dom_full, d_short in domains.items():
    metric_cols.extend([
        f"{d_short}_pred_acc", f"{d_short}_pred_f1",
        f"{d_short}_arg_acc",  f"{d_short}_arg_f1",
    ])

df = df[["run", "model_type"] + metric_cols]

df
